# Colab Automation: Train + Drive Backup + Resume Check

This notebook automates the full training flow for `collab_scripts`:
1. Mount Google Drive
2. Clone/Bootstrap from GitHub
3. Update `pipeline_config.json` paths
4. Run split -> train -> evaluate -> export
5. Verify backups are written to Drive
6. Check whether resume is ready and working

In [ ]:
from pathlib import Path
import json
import re
import subprocess
import textwrap

# ---- Required values ----
GITHUB_REPO_URL = "https://github.com/<org>/<repo>.git"
GITHUB_BRANCH = "main"
REPO_DIR = Path("/content/intruder_detection_system")

# Raw data location (class folders: fight/theft/intrusion/normal)
RAW_DATASET_DIR = Path("/content/raw_dataset")

# Paths for outputs/checkpoints
DATASET_DIR = Path("/content/dataset")
ARTIFACT_DIR = Path("/content/artifacts")
DRIVE_CHECKPOINT_DIR = Path("/content/drive/MyDrive/action_model_checkpoints")
LOCAL_CHECKPOINT_DIR = Path("/content/checkpoints")

# Execution toggles
RUN_PIPELINE_NOW = True
RUN_RESUME_COMMAND_CHECK = True

def run_cmd(command, cwd=None, check=True):
    print("$", " \
,
,
,
Config initialized. Update GITHUB_REPO_URL before running next cells.")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
print("Google Drive mounted.")

In [ ]:
if not (REPO_DIR / ".git").exists():
    run_cmd([
        "git",
        "clone",
        "--branch",
        GITHUB_BRANCH,
        "--single-branch",
        GITHUB_REPO_URL,
        str(REPO_DIR),
    ])
else:
    print(f"Repository already exists at {REPO_DIR}; pulling latest...")
    run_cmd(["git", "pull"], cwd=REPO_DIR)

bootstrap_candidates = [
    REPO_DIR / "backend/collab_scripts/bootstrap_colab.sh",
    REPO_DIR / "collab_scripts/bootstrap_colab.sh",
]
bootstrap_path = next((p for p in bootstrap_candidates if p.exists()), None)
if bootstrap_path is None:
    raise FileNotFoundError("Could not find bootstrap_colab.sh in cloned repo")

run_cmd(["bash", str(bootstrap_path), str(REPO_DIR)])
print("Clone + bootstrap complete.")

In [ ]:
config_candidates = [
    REPO_DIR / "backend/collab_scripts/pipeline_config.json",
    REPO_DIR / "collab_scripts/pipeline_config.json",
]
config_path = next((p for p in config_candidates if p.exists()), None)
if config_path is None:
    raise FileNotFoundError("pipeline_config.json not found in repo")

config = json.loads(config_path.read_text(encoding="utf-8"))
config.setdefault("paths", {})
config["paths"]["raw_dataset_dir"] = str(RAW_DATASET_DIR)
config["paths"]["dataset_dir"] = str(DATASET_DIR)
config["paths"]["checkpoint_dir"] = str(LOCAL_CHECKPOINT_DIR)
config["paths"]["drive_checkpoint_dir"] = str(DRIVE_CHECKPOINT_DIR)
config["paths"]["artifact_dir"] = str(ARTIFACT_DIR)
config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")

print(f"Updated config: {config_path}")
print(json.dumps(config["paths"], indent=2))

In [ ]:
expected_classes = ["fight", "theft", "intrusion", "normal"]
missing = [c for c in expected_classes if not (RAW_DATASET_DIR / c).exists()]
if missing:
    raise FileNotFoundError(f"Missing class folders under {RAW_DATASET_DIR}: {missing}")
print("Dataset class folders found.")

In [ ]:
if RUN_PIPELINE_NOW:
    run_script_candidates = [
        REPO_DIR / "backend/collab_scripts/colab_run_training.sh",
        REPO_DIR / "collab_scripts/colab_run_training.sh",
    ]
    run_script = next((p for p in run_script_candidates if p.exists()), None)
    if run_script is None:
        raise FileNotFoundError("colab_run_training.sh not found in repo")

    completed = run_cmd([
        "bash",
        str(run_script),
        str(REPO_DIR),
        "--config",
        str(config_path.relative_to(REPO_DIR)),
    ], cwd=REPO_DIR)
    print(completed.stdout[-2000:])
else:
    print("RUN_PIPELINE_NOW is False. Skipping pipeline run.")

In [ ]:
drive_last = DRIVE_CHECKPOINT_DIR / "last.pt"
drive_best = DRIVE_CHECKPOINT_DIR / "best.pt"

print(f"Drive last.pt exists: {drive_last.exists()}")
print(f"Drive best.pt exists: {drive_best.exists()}")
if drive_last.exists():
    print(f"Drive last.pt size: {drive_last.stat().st_size} bytes")
    print(f"Drive last.pt mtime: {drive_last.stat().st_mtime}")
if drive_best.exists():
    print(f"Drive best.pt size: {drive_best.stat().st_size} bytes")
    print(f"Drive best.pt mtime: {drive_best.stat().st_mtime}")

In [ ]:
import torch

local_last = LOCAL_CHECKPOINT_DIR / "last.pt"
checkpoint_for_check = local_last if local_last.exists() else (DRIVE_CHECKPOINT_DIR / "last.pt")

if checkpoint_for_check.exists():
    ckpt = torch.load(checkpoint_for_check, map_location="cpu")
    start_epoch = int(ckpt.get("epoch", -1)) + 1
    print(f"Resume checkpoint found: {checkpoint_for_check}")
    print(f"Next start epoch: {start_epoch}")
    if start_epoch > 0:
        print("Resume feature status: WORKING (checkpoint state is available).")
    else:
        print("Resume feature status: NOT READY (checkpoint has no progress yet).")
else:
    print("Resume feature status: NOT WORKING YET (no last.pt found in local or Drive).")

if RUN_RESUME_COMMAND_CHECK:
    train_cmd = [
        "python",
        "-m",
        "collab_scripts.train_action_model",
        "--config",
        str(config_path.relative_to(REPO_DIR)),
        "--auto-resume",
    ]
    work_root = REPO_DIR / "backend" if (REPO_DIR / "backend/collab_scripts").exists() else REPO_DIR
    result = run_cmd(train_cmd, cwd=work_root, check=False)
    resume_match = re.search(r"Starting at epoch:\s*(\d+)", result.stdout)
    if resume_match:
        resumed_epoch = int(resume_match.group(1))
        print(f"Resume command output epoch: {resumed_epoch}")
        print("Resume command check: WORKING" if resumed_epoch > 0 else "Resume command check: FRESH START")
    else:
        print("Resume command check: Could not find resume marker in stdout.")
    print("--- train_action_model stdout (tail) ---")
    print(result.stdout[-2000:])
    print("--- train_action_model stderr (tail) ---")
    print(result.stderr[-1000:])